<a href="https://colab.research.google.com/github/Architag1503/Colab/blob/main/MarkovDecisionProcess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import numpy as np
import zipfile
import os

# =========================
# 1. Load Dataset
# =========================
def load_data(zip_path):
    extract_path = "/content/data"

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    # Find CSV file automatically
    csv_file = None
    for file in os.listdir(extract_path):
        if file.endswith(".csv"):
            csv_file = os.path.join(extract_path, file)
            break

    if csv_file is None:
        raise Exception("No CSV file found in ZIP!")

    df = pd.read_csv(csv_file)

    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(by='Date')
    df.reset_index(drop=True, inplace=True)

    return df


# =========================
# 2. Feature Engineering (STATE)
# =========================
def create_states(df):

    # Price trend
    df['price_diff'] = df['Close'] - df['Open']

    def price_trend(x):
        if x > 0.5:
            return 1
        elif x < -0.5:
            return -1
        else:
            return 0

    df['price_trend'] = df['price_diff'].apply(price_trend)

    # Sentiment state
    def sentiment_cat(x):
        if x > 0.2:
            return 1
        elif x < -0.2:
            return -1
        else:
            return 0

    df['sentiment_state'] = df['Sentiment_Score'].apply(sentiment_cat)

    # Combine states
    df['state'] = list(zip(df['price_trend'], df['sentiment_state']))

    # Encode states
    unique_states = df['state'].unique()
    state_map = {s: i for i, s in enumerate(unique_states)}

    df['state_id'] = df['state'].map(state_map)

    return df, state_map


# =========================
# 3. Reward Function
# =========================
def get_reward(action, target):
    if action == 1:  # Buy
        return target
    elif action == 2:  # Sell
        return -target
    else:  # Hold
        return -0.1


# =========================
# 4. Q-Learning (MDP Solver)
# =========================
def train_q_learning(df, num_states, num_actions=3,
                     alpha=0.1, gamma=0.9, epsilon=0.1, episodes=20):

    Q = np.zeros((num_states, num_actions))

    for ep in range(episodes):
        for i in range(len(df) - 1):

            s = df.loc[i, 'state_id']
            next_s = df.loc[i + 1, 'state_id']
            target = df.loc[i, 'Target']

            # Epsilon-greedy
            if np.random.rand() < epsilon:
                action = np.random.choice(num_actions)
            else:
                action = np.argmax(Q[s])

            reward = get_reward(action, target)

            # Q update
            Q[s, action] += alpha * (
                reward + gamma * np.max(Q[next_s]) - Q[s, action]
            )

    return Q


# =========================
# 5. Policy Extraction
# =========================
def get_policy(Q):
    return np.argmax(Q, axis=1)


# =========================
# 6. Evaluation
# =========================
def evaluate_policy(df, policy):
    total_reward = 0
    actions_list = []

    for i in range(len(df) - 1):
        s = df.loc[i, 'state_id']
        action = policy[s]
        target = df.loc[i, 'Target']

        reward = get_reward(action, target)
        total_reward += reward

        actions_list.append(action)

    return total_reward, actions_list


# =========================
# 7. Run MDP per Company
# =========================
def run_per_company(df):

    results = {}
    companies = df['Company'].unique()

    for company in companies:

        print(f"\n🔹 Processing: {company}")

        company_df = df[df['Company'] == company].copy()
        company_df.reset_index(drop=True, inplace=True)

        if len(company_df) < 10:
            print("Skipping (not enough data)")
            continue

        # Create states
        company_df, state_map = create_states(company_df)

        # Train Q-learning
        Q = train_q_learning(company_df, len(state_map))

        # Extract policy
        policy = get_policy(Q)

        # Evaluate
        total_reward, actions = evaluate_policy(company_df, policy)

        # Count actions
        buy = actions.count(1)
        sell = actions.count(2)
        hold = actions.count(0)

        results[company] = {
            "Total Reward": total_reward,
            "Buy": buy,
            "Sell": sell,
            "Hold": hold,
            "States": len(state_map)
        }

    return results


# =========================
# MAIN
# =========================
if __name__ == "__main__":

    zip_path = "/content/drive/MyDrive/dataset/archive (3).zip"

    df = load_data(zip_path)

    results = run_per_company(df)

    print("\n================ FINAL RESULTS ================\n")

    for company, res in results.items():
        print(f"📊 Company: {company}")
        print(f"   Total Reward: {res['Total Reward']}")
        print(f"   Buy Decisions: {res['Buy']}")
        print(f"   Sell Decisions: {res['Sell']}")
        print(f"   Hold Decisions: {res['Hold']}")
        print(f"   Unique States: {res['States']}")
        print("-" * 50)


🔹 Processing: Apple

🔹 Processing: Amazon

🔹 Processing: Tesla

================ FINAL RESULTS ================

📊 Company: Apple
   Total Reward: -162.49999999999775
   Buy Decisions: 1403
   Sell Decisions: 1220
   Hold Decisions: 1365
   Unique States: 3
--------------------------------------------------
📊 Company: Amazon
   Total Reward: 51
   Buy Decisions: 3757
   Sell Decisions: 0
   Hold Decisions: 0
   Unique States: 3
--------------------------------------------------
📊 Company: Tesla
   Total Reward: 1131
   Buy Decisions: 0
   Sell Decisions: 4321
   Hold Decisions: 0
   Unique States: 3
--------------------------------------------------
